## Registro de experimentos del pipeline de Machine Learning

### Proyecto:
Pacientes con problemas de hígado - India

**Autor:** Mariana Bedoya Arismendy

### Fecha:
2026-08-19

### Incidencia:
12 - Selección del mejor modelo

### Descripción:

Este notebook replica el entrenamiento del primer modelo (Random Forest, ver
`03-first_model-mba-2026-08-19.ipynb`) añadiendo el **registro de experimentos
con MLflow**: cada ejecución queda documentada con sus hiperparámetros,
métricas, etiquetas y el propio modelo como artefacto, de modo que cualquier
resultado sea trazable y comparable con ejecuciones futuras.

El flujo: carga de datos → limpieza → preprocesamiento → división → ajuste de
hiperparámetros con `GridSearchCV` → evaluación → registro de la ejecución en
MLflow → guardado del modelo → carga del modelo registrado para predecir.

## 📚 Importar librerías

In [1]:
# librerías base para ciencia de datos
import os
import sys
from pathlib import Path

import mlflow
import pandas as pd
import sklearn as sk
from joblib import dump, load
from mlflow.models import infer_signature
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# mlflow >= 3 deshabilitó el backend de archivos (carpeta ./mlruns) por defecto;
# lo reactivamos explícitamente para mantener el tracking local del proyecto
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

## 💾 Cargar datos

Cargamos el mismo archivo Parquet de la etapa intermedia que usaron el EDA, el
Feature Engineering, el baseline y la selección de modelos: las 11 variables
clínicas con los tipos de datos ya corregidos.

In [2]:
DATA_DIR = Path.cwd().resolve().parents[1] / "data"
DATA_PATH = DATA_DIR / "02_intermediate/Pacientes_porblemas_higado_india_type_fixed.parquet"

assert DATA_PATH.exists(), f"No se encuentra el archivo del proyecto: {DATA_PATH}"

pacientes_df = pd.read_parquet(DATA_PATH, engine="pyarrow")

print(f"Dataset cargado: {pacientes_df.shape[0]} filas y {pacientes_df.shape[1]} columnas")
pacientes_df.head()

Dataset cargado: 663 filas y 11 columnas


,Age,Gender,Total_Bilirubin,Direct_Bilirubin,Alkaline_Phosphotase,Alamine_Aminotransferase,Aspartate_Aminotransferase,Total_Protiens,Albumin,Albumin_and_Globulin_Ratio,Dataset
0,65,Female,0.7,0.1,187.0,16.0,18.0,6.8,3.3,0.9,1
1,62,Male,10.9,5.5,699.0,64.0,100.0,7.5,3.2,0.74,1
2,62,Male,7.3,4.1,490.0,60.0,68.0,7.0,3.3,0.89,1
3,58,Male,1.0,0.4,182.0,14.0,20.0,6.8,3.4,1.0,1
4,72,Male,3.9,2.0,195.0,27.0,59.0,7.3,2.4,0.4,1


In [3]:
# versiones de las librerías para reproducibilidad
print("Versión de Python:", sys.version)
print("Versión de pandas:", pd.__version__)
print("Versión de scikit-learn:", sk.__version__)
print("Versión de mlflow:", mlflow.__version__)

Versión de Python: 3.12.11 (main, Jul  1 2025, 05:29:09) [GCC 12.2.0]
Versión de pandas: 3.0.5
Versión de scikit-learn: 1.9.0
Versión de mlflow: 3.15.1


## 👷 Preparación de los datos

Aplicamos la misma secuencia de limpieza de las etapas anteriores (duplicados
exactos → filas sin etiqueta → bilirrubinas inconsistentes, siempre antes del
split) y recodificamos la variable objetivo `Dataset` como 1 (problemas de
hígado) / 0 (sin problemas), de modo que este notebook trabaje sobre exactamente
los mismos 585 registros y la misma partición 468/117 que el baseline y la
selección de modelos.

In [4]:
# 1. Duplicados exactos: antes del split para evitar registros idénticos
#    en entrenamiento y prueba.
filas_iniciales = len(pacientes_df)
limpio_df = pacientes_df.drop_duplicates()
print(f"Tras eliminar duplicados: {len(limpio_df)} filas (-{filas_iniciales - len(limpio_df)})")

# 2. Registros sin etiqueta en el objetivo: la variable objetivo no se imputa.
filas_sin_etiqueta = int(limpio_df["Dataset"].isna().sum())
limpio_df = limpio_df[limpio_df["Dataset"].notna()]
print(f"Tras descartar filas sin etiqueta: {len(limpio_df)} filas (-{filas_sin_etiqueta})")

# 3. Bilirrubinas inconsistentes (directa mayor que la total): errores de captura.
inconsistente = (limpio_df["Direct_Bilirubin"] > limpio_df["Total_Bilirubin"]).fillna(False)
limpio_df = limpio_df[~inconsistente]
print(f"Tras descartar filas inconsistentes: {len(limpio_df)} filas (-{int(inconsistente.sum())})")

# objetivo binario: 1 = problemas de hígado ('1'), 0 = sin problemas ('2')
pacientes_features = limpio_df.copy()
pacientes_features["Dataset"] = (pacientes_features["Dataset"] == "1").astype(int)

pacientes_features.info()

Tras eliminar duplicados: 603 filas (-60)
Tras descartar filas sin etiqueta: 588 filas (-15)
Tras descartar filas inconsistentes: 585 filas (-3)
<class 'pandas.DataFrame'>
Index: 585 entries, 0 to 662
Data columns (total 11 columns):
 #   Column                      Non-Null Count  Dtype   
---  ------                      --------------  -----   
 0   Age                         584 non-null    Int64   
 1   Gender                      581 non-null    category
 2   Total_Bilirubin             582 non-null    Float64 
 3   Direct_Bilirubin            581 non-null    Float64 
 4   Alkaline_Phosphotase        576 non-null    Float64 
 5   Alamine_Aminotransferase    573 non-null    Float64 
 6   Aspartate_Aminotransferase  577 non-null    Float64 
 7   Total_Protiens              578 non-null    Float64 
 8   Albumin                     583 non-null    Float64 
 9   Albumin_and_Globulin_Ratio  581 non-null    Float64 
 10  Dataset                     585 non-null    int64   
dtypes: Floa

## 👨‍🏭 Ingeniería de características (preprocesamiento)

Usamos el mismo preprocesamiento simple de la comparación de algoritmos del
notebook 02, común a todos los modelos de esa etapa:

- **Numéricas** (edad y las ocho pruebas hepáticas): imputación por la mediana
  y estandarización.
- **Categóricas** (`Gender`): imputación por la moda y codificación one-hot.

La imputación vive dentro del pipeline: así se calcula solo con los datos de
entrenamiento y se aplica igual a datos nuevos en producción, sin fuga de
información.

In [5]:
# columnas numéricas: edad y las ocho pruebas hepáticas
cols_numeric = [
    "Age",
    "Total_Bilirubin",
    "Direct_Bilirubin",
    "Alkaline_Phosphotase",
    "Alamine_Aminotransferase",
    "Aspartate_Aminotransferase",
    "Total_Protiens",
    "Albumin",
    "Albumin_and_Globulin_Ratio",
]

# columnas categóricas nominales
cols_categoric = ["Gender"]

In [6]:
numeric_pipe = Pipeline(
    steps=[
        ("imputador_mediana", SimpleImputer(strategy="median")),
        ("escalado", StandardScaler()),
    ]
)

categorical_pipe = Pipeline(
    steps=[
        ("imputador_moda", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="infrequent_if_exist", sparse_output=False)),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("numericas", numeric_pipe, cols_numeric),
        ("categoricas", categorical_pipe, cols_categoric),
    ],
    verbose_feature_names_out=True,
)

# salida como DataFrame de pandas para conservar los nombres de las columnas
preprocessor.set_output(transform="pandas")
preprocessor

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numericas', ...), ('categoricas', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``

## ✂️ División entrenamiento / prueba

División 80 %/20 % estratificada por el objetivo con `random_state=42`: la
misma partición de los notebooks del baseline, la selección de modelos y el
AutoML (468 pacientes para entrenamiento y 117 para prueba).

In [7]:
SEED = 42

X_features = pacientes_features.drop("Dataset", axis="columns")
Y_target = pacientes_features["Dataset"]

# 80 % entrenamiento, 20 % prueba
x_train, x_test, y_train, y_test = train_test_split(
    X_features, Y_target, test_size=0.2, stratify=Y_target, random_state=SEED
)

print(f"Entrenamiento: {x_train.shape[0]} filas ({int(y_train.sum())} con problemas de hígado)")
print(f"Prueba: {x_test.shape[0]} filas ({int(y_test.sum())} con problemas de hígado)")

Entrenamiento: 468 filas (331 con problemas de hígado)
Prueba: 117 filas (83 con problemas de hígado)


## 👨‍🏭 Crear el pipeline

El pipeline completo empaqueta el preprocesamiento y el clasificador Random
Forest: se ajusta y evalúa como una sola pieza, y es lo que se persiste al
final del notebook.

In [8]:
data_model_pipeline = Pipeline(
    steps=[("preprocessor", preprocessor), ("model", RandomForestClassifier(random_state=SEED))]
)
data_model_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numericas', ...), ('categoricas', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, def

## Ajuste de hiperparámetros

Seleccionamos los mejores hiperparámetros del Random Forest con `GridSearchCV`
(5 folds), optimizando el recall de la clase positiva — pacientes con
problemas de hígado —, igual que en el notebook 02. La cuadrícula es la misma
que allí resultó ganadora.

### Random Forest

In [9]:
score = "recall"

hyperparameters = {
    "model__max_depth": [4, 5, 7, 9, 10],
    "model__max_features": [2, 3, 4, 5, 6, 7, 8, 9, 10, 11],
    "model__criterion": ["gini", "entropy"],
}


grid_search = GridSearchCV(
    data_model_pipeline,
    hyperparameters,
    cv=5,
    scoring=score,
    n_jobs=8,
)
grid_search.fit(x_train, y_train)

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__criterion': ['gini', 'entropy'], 'model__max_depth': [4, 5, ...], 'model__max_features': [2, 3, ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'recall'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",8
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose:

In [10]:
grid_search.best_params_

{'model__criterion': 'entropy',
 'model__max_depth': 4,
 'model__max_features': 2}

In [11]:
best_data_model_pipeline = grid_search.best_estimator_
best_data_model_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](10,)","['Age','Gender','Total_Bilirubin',...,'Total_Protiens','Albumin', 'Albumin_and_Globulin_Ratio']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,10
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numericas', ...), ('categoricas', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specif

In [12]:
type(best_data_model_pipeline)

sklearn.pipeline.Pipeline

### Evaluación

Calculamos las métricas sobre el conjunto de prueba; serán las que se registren
en la ejecución de MLflow.

In [13]:
y_pred = best_data_model_pipeline.predict(x_test)
y_proba = best_data_model_pipeline.predict_proba(x_test)[:, 1]
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
# la AUC se calcula con las probabilidades, no con las etiquetas predichas
auc = roc_auc_score(y_test, y_proba)
print(f"recall: {recall}")
print(f"precision: {precision}")
print(f"f1: {f1}")
print(f"auc: {auc}")

recall: 0.9036144578313253
precision: 0.7211538461538461
f1: 0.8021390374331551
auc: 0.7583274273564847


## Registrar el experimento con MLflow

Configuramos el tracking local (carpeta `mlruns/` en la raíz del proyecto) y
creamos el experimento `pacientes_higado_models`. Dentro de la ejecución
registramos:

- **Etiqueta** con la información del entrenamiento.
- **Hiperparámetros** ganadores de la búsqueda.
- **Métricas** de evaluación en prueba.
- **El modelo** como artefacto: el mismo pipeline en formato joblib que usa el
  proyecto, acompañado de su firma (tipos de entrada y salida) en JSON.

Usamos `mlflow-skinny`, la instalación ligera y oficial de MLflow para tracking,
que provee toda la API de ejecuciones, parámetros, métricas y artefactos sin el
servidor completo (innecesario en este proyecto).


In [14]:
mlflow.set_tracking_uri((Path.cwd().resolve().parents[1] / "mlruns").as_uri())

exp = mlflow.set_experiment(experiment_name="pacientes_higado_models")

# ruta del modelo del proyecto (formato joblib, igual que el notebook 03)
DATA_MODEL = Path.cwd().resolve().parents[1] / "models"
ruta_modelo_joblib = DATA_MODEL / "pacientes_higado_random_forest-v1.joblib"

with mlflow.start_run() as run:
    # registrar la etiqueta de entrenamiento
    mlflow.set_tag("Training Info", "Primer modelo: Random Forest")

    # registrar los hiperparámetros ganadores de la búsqueda
    mlflow.log_params(grid_search.best_params_)

    # registrar las métricas de evaluación en prueba
    mlflow.log_metrics(
        {
            "recall": recall,
            "precision": precision,
            "f1": f1,
            "auc": auc,
        }
    )

    # persistir el pipeline con joblib y registrarlo como artefacto del run
    dump(best_data_model_pipeline, ruta_modelo_joblib, protocol=5)
    mlflow.log_artifact(ruta_modelo_joblib, artifact_path="pacientes_higado_model")

    # documentar la firma del modelo (entradas y salidas esperadas) en JSON
    signature = infer_signature(x_train, best_data_model_pipeline.predict(x_train))
    mlflow.log_dict(
        {"signature": signature.to_dict()},
        artifact_file="pacientes_higado_model/signature.json",
    )

/workspaces/Pacientes_porblemas_higado_india/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


## 💾 Guardar el modelo

El pipeline quedó persistido en dos lugares equivalentes dentro de la ejecución
anterior: como artefacto de MLflow (en `mlruns/`) y como modelo del proyecto en
`models/` con el formato joblib usado en el notebook 03.


In [15]:
print(f"Modelo del proyecto: {ruta_modelo_joblib}")
print(f"Carpeta de tracking: {Path.cwd().resolve().parents[1] / 'mlruns'}")

Modelo del proyecto: /workspaces/Pacientes_porblemas_higado_india/models/pacientes_higado_random_forest-v1.joblib
Carpeta de tracking: /workspaces/Pacientes_porblemas_higado_india/mlruns


## Cargar el modelo para hacer predicciones

Verificamos que el modelo guardado se puede recargar con joblib y predecir
sobre datos nuevos sin reproducir nada del entrenamiento.


In [16]:
load_model = load(ruta_modelo_joblib)

predictions = load_model.predict(x_test)
predictions[:12]

array([1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1])

## 📖 Referencias

- <https://joserzapata.github.io/courses/python-ciencia-datos/ml/>
- <https://joserzapata.github.io/courses/python-ciencia-datos/clasificacion/>
- Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow: Concepts, Tools, and Techniques to Build Intelligent Systems 2nd Edition - Aurélien Géron
- Dataset ILPD (Indian Liver Patient Dataset):
  <https://archive.ics.uci.edu/dataset/225/ilpd+indian+liver+patient+dataset>